# WP5 — Multi-Agent Safety Protocols Demo
**Prometheus v0.97**

This notebook demonstrates the WP5 multi-agent communication and coordination layer:

1. **HMAC-SHA256 message authentication** — sign, verify, tamper detection
2. **Replay protection** — duplicate message rejection
3. **Safety-gated coordination** — MCSSupervisor gates every plan
4. **Priority auction** — resource deconfliction with Gini fairness metric
5. **Quarantine** — persistent violators isolated
6. **Full benchmark** — cooperative / adversarial / mixed scenarios


In [ ]:
# Colab setup — clone repo if needed
import sys, os
if 'google.colab' in sys.modules:
    os.system('git clone https://github.com/prometheus-ai/Prometheus_v0_PoC /content/Prometheus_v0_PoC 2>/dev/null || true')
    sys.path.insert(0, '/content/Prometheus_v0_PoC')
else:
    sys.path.insert(0, os.path.abspath('..'))

import warnings
warnings.filterwarnings('ignore')
print('Environment ready.')

In [ ]:
import json
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

from prometheus.multi_agent_comms import (
    AgentIdentity, Message, MessageBus, AgentChannel,
    CoordinatorAgent, MsgType, make_session,
)
from prometheus.safety.mcs_supervisor import MCSSupervisor
from benchmarks.multi_agent_benchmark import MultiAgentBenchmark

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
print('Imports OK.')

---
## 1 — HMAC-SHA256 Message Authentication

In [ ]:
SHARED_SECRET = b'prometheus-demo-key'

alice = AgentIdentity('alice', SHARED_SECRET)
bob   = AgentIdentity('bob',   SHARED_SECRET)

# Alice signs a message
data = b'{"msg": "hello from alice", "seq": 1}'
sig  = alice.sign(data)
print(f'Signature (first 16 chars): {sig[:16]}...')

# Alice verifies her own signature
print(f'Alice verifies own sig : {alice.verify(data, sig)}')

# Bob verifies (same shared secret)
print(f'Bob   verifies alice sig: {bob.verify(data, sig)}')

# Tampered data fails
tampered = b'{"msg": "TAMPERED", "seq": 1}'
print(f'Tampered data fails     : {not alice.verify(tampered, sig)}')

# Wrong key fails
eve = AgentIdentity('eve', b'wrong-key')
print(f'Wrong key fails         : {not eve.verify(data, sig)}')

---
## 2 — MessageBus: Delivery, Replay & Quarantine

In [ ]:
# ── 2a. Basic delivery
bus = MessageBus({'alice': alice, 'bob': bob})
ch_alice = AgentChannel(alice, bus)
ch_alice.send('bob', MsgType.HEARTBEAT, {'ping': True})
msgs = bus.receive('bob')
print(f'Bob received {len(msgs)} message(s): type={msgs[0].msg_type}')

# ── 2b. Tampered message dropped
ch_alice.send('bob', MsgType.PLAN, {'code': 'x = 1'})
bus._inboxes['bob'][0].payload['code'] = 'EVIL CODE'   # tamper in place
msgs = bus.receive('bob')
print(f'Tampered message count received: {len(msgs)} (expected 0)')

# ── 2c. Replay protection
bus2   = MessageBus({'alice': alice, 'bob': bob})
ch_a2  = AgentChannel(alice, bus2)
ch_a2.send('bob', MsgType.HEARTBEAT, {})
# Mark next seq_no as already seen
next_seq = alice._seq_counter + 1
bus2._seen_seqs.add(('alice', next_seq))
ok = ch_a2.send('bob', MsgType.HEARTBEAT, {})
print(f'Replay rejected: {not ok}')

# ── 2d. Quarantine
bus3 = MessageBus({'alice': alice, 'bob': bob})
bus3.quarantine('alice')
ch_a3 = AgentChannel(alice, bus3)
ok = ch_a3.send('bob', MsgType.PLAN, {'code': 'y = 2'})
print(f'Quarantined send rejected: {not ok}')

---
## 3 — Safety-Gated Coordination (MCSSupervisor)

In [ ]:
agent_ids = ['alpha', 'beta', 'rogue']
session = make_session(
    agent_ids           = agent_ids,
    shared_secret       = SHARED_SECRET,
    violation_threshold = 2,
)
session.start()

# Safe plans
session.channels['alpha'].send_plan('coordinator', {'code': 'result = 6 * 7'})
session.channels['beta'].send_plan('coordinator',  {'code': 'data = [i**2 for i in range(5)]'})

# Unsafe plan from rogue
session.channels['rogue'].send_plan('coordinator', {'code': 'import os\nos.system("echo pwned")'})

result = session.coordinator.run_round()

print(f'Plans received : {result.n_plans_received}')
print(f'Plans approved : {result.n_plans_approved}')
print(f'Plans rejected : {result.n_plans_rejected}')
print(f'Safety violations:')
for v in result.safety_violations:
    print(f'  [{v["agent_id"]}] {v["violation_type"]} (severity {v["severity"]})')

# Second unsafe plan → quarantine
session.channels['rogue'].send_plan('coordinator', {'code': 'import subprocess\nsubprocess.run(["ls"])'})
result2 = session.coordinator.run_round()

print(f'\nAfter round 2:')
print(f'  Rogue quarantined: {"rogue" in session.bus.quarantined_agents}')
print(f'  Total quarantined: {list(session.bus.quarantined_agents)}')

In [ ]:
# Visualise plan outcomes
fig, ax = plt.subplots(figsize=(7, 4))

categories = ['Approved\n(round 1)', 'Rejected\n(round 1)', 'Quarantined\n(after round 2)']
values     = [result.n_plans_approved, result.n_plans_rejected, len(session.bus.quarantined_agents)]
colors     = ['#2ecc71', '#e74c3c', '#e67e22']

bars = ax.bar(categories, values, color=colors, edgecolor='white', linewidth=1.5, width=0.5)
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.05,
            str(val), ha='center', va='bottom', fontweight='bold')

ax.set_ylabel('Count')
ax.set_title('CoordinatorAgent Plan Safety Gating\n(2 benign agents + 1 rogue)', fontweight='bold')
ax.set_ylim(0, max(values) + 1)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

---
## 4 — Priority Auction & Fairness (Gini Coefficient)

In [ ]:
from benchmarks.multi_agent_benchmark import _gini

# Fresh session for auction demo
auc_session = make_session(
    ['agent_1', 'agent_2', 'agent_3', 'agent_4'],
    shared_secret = SHARED_SECRET,
)
auc_session.start()

rng = np.random.default_rng(42)
resources  = ['gpu_slot', 'cpu_slot', 'memory']
win_counts = {aid: 0 for aid in auc_session.agent_ids}

N_AUC_ROUNDS = 10
for _ in range(N_AUC_ROUNDS):
    for aid in auc_session.agent_ids:
        resource = resources[int(rng.integers(0, len(resources)))]
        bid_val  = float(rng.uniform(1.0, 10.0))
        auc_session.channels[aid].bid('coordinator', resource, bid_val)

    result = auc_session.coordinator.run_round()
    for resource, winner in result.auction_awards.items():
        win_counts[winner] = win_counts.get(winner, 0) + 1

gini = _gini(list(win_counts.values()))
print(f'Win counts: {win_counts}')
print(f'Gini coefficient: {gini:.3f}  (0=perfectly fair, 1=monopoly)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: win distribution
ax = axes[0]
agents = list(win_counts.keys())
wins   = [win_counts[a] for a in agents]
palette = plt.cm.Set2(np.linspace(0, 1, len(agents)))
bars = ax.bar(agents, wins, color=palette, edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, wins):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.1,
            str(val), ha='center', va='bottom', fontweight='bold')
ax.set_title(f'Auction Win Distribution\n(Gini = {gini:.3f})', fontweight='bold')
ax.set_ylabel('Wins')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Right: Lorenz curve
ax2 = axes[1]
sorted_wins = sorted(wins)
n = len(sorted_wins)
cum = np.cumsum(sorted_wins) / max(sum(sorted_wins), 1)
lorenz_x = np.linspace(0, 1, n)
ax2.plot([0] + list(lorenz_x), [0] + list(cum), 'b-o', label='Lorenz curve', linewidth=2)
ax2.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Perfect equality')
ax2.fill_between([0] + list(lorenz_x), [0] + list(cum), [0] + list(lorenz_x),
                 alpha=0.15, color='blue')
ax2.set_title('Lorenz Curve — Auction Fairness', fontweight='bold')
ax2.set_xlabel('Cumulative fraction of agents')
ax2.set_ylabel('Cumulative fraction of wins')
ax2.legend()
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

---
## 5 — Full Benchmark: Cooperative / Adversarial / Mixed

In [ ]:
bench   = MultiAgentBenchmark(n_rounds=5, n_resources=3, seed=42)
results = bench.run_all()

print(bench.summary_table(results))
print()
for r in results:
    print(f'  [{r.scenario}] {r.notes}')

In [ ]:
scenarios     = [r.scenario      for r in results]
safe_rates    = [r.safe_rate * 100 for r in results]
violations    = [r.n_violations  for r in results]
quarantined   = [r.n_quarantined for r in results]
ginis         = [r.auction_gini  for r in results]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
palette   = ['#2ecc71', '#e74c3c', '#e67e22']

def _bar(ax, values, title, ylabel, color_list, fmt=None):
    bars = ax.bar(scenarios, values, color=color_list, edgecolor='white', linewidth=1.5)
    for bar, val in zip(bars, values):
        label = f'{val:.1f}' if fmt == 'f' else str(val)
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01 * max(values + [1]),
                label, ha='center', va='bottom', fontsize=10, fontweight='bold')
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel(ylabel)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

_bar(axes[0, 0], safe_rates,  'Plan Approval Rate (%)',    '%',              palette, 'f')
_bar(axes[0, 1], violations,  'Safety Violations',          'Count',          palette)
_bar(axes[1, 0], quarantined, 'Agents Quarantined',         'Count',          palette)
_bar(axes[1, 1], ginis,       'Auction Gini Coefficient\n(0=fair, 1=monopoly)', 'Gini', palette, 'f')

for ax in axes.flat:
    ax.set_xticks(range(len(scenarios)))
    ax.set_xticklabels([s.capitalize() for s in scenarios])

fig.suptitle('Multi-Agent Safety Benchmark — Prometheus v0.97 (WP5)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

---
## 6 — Round-by-Round Timeline (Adversarial Scenario)

In [ ]:
adv_result   = next(r for r in results if r.scenario == 'adversarial')
round_nums   = [rr.round_no         for rr in adv_result.round_results]
approved     = [rr.n_plans_approved for rr in adv_result.round_results]
rejected     = [rr.n_plans_rejected for rr in adv_result.round_results]
quarantines  = [rr.n_quarantines    for rr in adv_result.round_results]

fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(round_nums))
w = 0.3

ax.bar(x - w, approved,    width=w, label='Approved', color='#2ecc71', edgecolor='white')
ax.bar(x,     rejected,    width=w, label='Rejected', color='#e74c3c', edgecolor='white')
ax.bar(x + w, quarantines, width=w, label='New quarantines', color='#e67e22', edgecolor='white')

ax.set_xticks(x)
ax.set_xticklabels([f'Round {r}' for r in round_nums])
ax.set_ylabel('Plans / agents')
ax.set_title('Adversarial Scenario — Round-by-Round Timeline', fontweight='bold')
ax.legend()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

print('\nViolation details:')
for rr in adv_result.round_results:
    for v in rr.safety_violations:
        print(f'  Round {rr.round_no} [{v["agent_id"]}] {v["violation_type"]} (sev={v["severity"]})')

---
## Summary

| Property | Mechanism | Status |
|----------|-----------|--------|
| Message authenticity | HMAC-SHA256 per-message signature | Implemented |
| Replay protection | `(sender_id, seq_no)` seen-set + nonce | Implemented |
| Tamper detection | Signature verification on receive | Implemented |
| Plan safety gating | MCSSupervisor in every coord round | Implemented |
| DoS mitigation | Quarantine after threshold violations | Implemented |
| Resource deconfliction | Priority (highest-bid-wins) auction | Implemented |
| Fairness monitoring | Gini coefficient over win distribution | Implemented |

**Test coverage**: 52 tests, all passing (`pytest tests/test_multi_agent.py -v`)

**Files**:
- `prometheus/multi_agent_comms.py` — core protocol module
- `benchmarks/multi_agent_benchmark.py` — benchmark runner
- `tests/test_multi_agent.py` — 52-test suite
- `MULTI_AGENT_SAFETY.md` — threat model and design documentation
